# Aula 22 — Normalização quando apropriado

Laboratório reproduzível em NumPy puro para padronização de entradas, BatchNorm e LayerNorm densas.

**Dependências mínimas:** Python 3.11, NumPy 1.26 e Matplotlib 3.8.  
**Reprodutibilidade:** dados sintéticos, seed fixa `20260922`, sem rede, segredos ou credenciais.

O notebook usa `float64` para tornar as verificações numéricas mais sensíveis. Os outputs são removidos do arquivo versionado depois que uma cópia é executada integralmente.

In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260922
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)
warnings.filterwarnings("error")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")
assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)
assert tuple(map(int, matplotlib.__version__.split(".")[:2])) >= (3, 8)

## 1. Padronização é um objeto ajustado no treino

As estatísticas são vetores por feature. `keepdims=True` preserva o contrato `(1, D)` e evita broadcasting acidental.

In [ ]:
def fit_standardizer(X, eps=1e-8):
    X = np.asarray(X, dtype=np.float64)
    return {
        "mean": X.mean(axis=0, keepdims=True),
        "var": X.var(axis=0, keepdims=True),
        "eps": float(eps),
    }


def transform_standardizer(X, state):
    return (np.asarray(X, dtype=np.float64) - state["mean"]) / np.sqrt(
        state["var"] + state["eps"]
    )


X_train_input = np.column_stack([
    rng.normal(40.0, 12.0, 400),
    rng.normal(4_000.0, 900.0, 400),
    rng.normal(8.0, 2.0, 400),
])
X_holdout_input = np.column_stack([
    rng.normal(65.0, 12.0, 100),
    rng.normal(8_000.0, 900.0, 100),
    rng.normal(14.0, 2.0, 100),
])

input_state = fit_standardizer(X_train_input)
Z_train_input = transform_standardizer(X_train_input, input_state)
Z_holdout_input = transform_standardizer(X_holdout_input, input_state)

train_mean_err = float(np.max(np.abs(Z_train_input.mean(axis=0))))
train_var_err = float(np.max(np.abs(Z_train_input.var(axis=0) - 1.0)))
print("erro máximo da média no treino:", f"{train_mean_err:.3e}")
print("erro máximo da variância no treino:", f"{train_var_err:.3e}")
print("média do holdout sob estatísticas do treino:", Z_holdout_input.mean(axis=0))
assert input_state["mean"].shape == (1, 3)
assert input_state["var"].shape == (1, 3)
assert train_mean_err < 1e-12
assert train_var_err < 1e-7

### Contraprova: ajustar antes do split vaza a distribuição reservada

O holdout foi gerado com deslocamento proposital. Ele deve ser apenas transformado. A célula abaixo mede quanto um ajuste contaminado mudaria até os valores do treino.

In [ ]:
leaked_state = fit_standardizer(np.vstack([X_train_input, X_holdout_input]))
Z_train_leaked = transform_standardizer(X_train_input, leaked_state)
leakage_shift = float(np.max(np.abs(Z_train_leaked - Z_train_input)))

print("maior alteração no treino causada pelo vazamento:", f"{leakage_shift:.6f}")
assert leakage_shift > 0.5
assert not np.allclose(leaked_state["mean"], input_state["mean"])
assert np.all(np.isfinite(Z_train_leaked))

## 2. Escalas e condicionamento

Criamos uma regressão com duas features em escalas `1` e `1000`. Cada versão usa um passo estável derivado do maior autovalor de sua Hessiana de treino. O teste permanece reservado até o fim.

In [ ]:
def add_bias(X):
    return np.column_stack([X, np.ones(len(X))])


def gd_quadratic(X, y, steps=250):
    A = add_bias(X)
    H = A.T @ A / len(A)
    eigvals = np.linalg.eigvalsh(H)
    eta = 0.9 / eigvals[-1]
    w = np.zeros(A.shape[1], dtype=np.float64)
    history = []
    for _ in range(steps):
        residual = A @ w - y
        history.append(0.5 * np.mean(residual**2))
        w -= eta * (A.T @ residual) / len(A)
    return w, np.asarray(history), eta, float(eigvals[-1] / eigvals[0])


def mse(X, y, w):
    return float(np.mean((add_bias(X) @ w - y) ** 2))


rng_reg = np.random.default_rng(SEED + 1)
X_reg = np.column_stack([rng_reg.normal(size=512), 1_000 * rng_reg.normal(size=512)])
y_reg = 3.0 * X_reg[:, 0] + 0.002 * X_reg[:, 1] + rng_reg.normal(0, 0.1, 512)
X_reg_test = np.column_stack([rng_reg.normal(size=256), 1_000 * rng_reg.normal(size=256)])
y_reg_test = 3.0 * X_reg_test[:, 0] + 0.002 * X_reg_test[:, 1] + rng_reg.normal(0, 0.1, 256)

reg_state = fit_standardizer(X_reg)
X_reg_std = transform_standardizer(X_reg, reg_state)
X_reg_test_std = transform_standardizer(X_reg_test, reg_state)

w_raw, loss_raw, eta_raw, cond_raw = gd_quadratic(X_reg, y_reg)
w_std, loss_std, eta_std, cond_std = gd_quadratic(X_reg_std, y_reg)
test_mse_raw = mse(X_reg_test, y_reg_test, w_raw)
test_mse_std = mse(X_reg_test_std, y_reg_test, w_std)

print(f"condicionamento bruto: {cond_raw:.3e}; padronizado: {cond_std:.6f}")
print(f"passo bruto: {eta_raw:.3e}; padronizado: {eta_std:.6f}")
print(f"MSE de teste bruto: {test_mse_raw:.6f}; padronizado: {test_mse_std:.6f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(loss_raw, label="entrada bruta")
ax.semilogy(loss_std, label="entrada padronizada")
ax.set(xlabel="passo", ylabel="0,5 × MSE de treino", title="Condicionamento e convergência")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

assert cond_raw > 1e5
assert cond_std < 1.3
assert loss_std[-1] < 0.01
assert test_mse_std < 0.02
assert test_mse_raw / test_mse_std > 100

## 3. Núcleo comum de BatchNorm e LayerNorm

O parâmetro `axis` define o grupo. `axis=0` reduz exemplos; `axis=1` reduz unidades. `eps` fica dentro da raiz.

In [ ]:
def norm_forward(X, gamma, beta, axis, eps=1e-5):
    X = np.asarray(X, dtype=np.float64)
    mean = X.mean(axis=axis, keepdims=True)
    var = X.var(axis=axis, keepdims=True)
    inv_std = 1.0 / np.sqrt(var + eps)
    xhat = (X - mean) * inv_std
    Y = gamma * xhat + beta
    cache = {"xhat": xhat, "inv_std": inv_std, "axis": axis}
    return Y, cache, mean, var


B, H = 12, 5
X = rng.normal(loc=np.arange(H), scale=np.linspace(0.5, 2.0, H), size=(B, H))
gamma = np.ones((1, H), dtype=np.float64)
beta = np.zeros((1, H), dtype=np.float64)

Y_bn, cache_bn, mean_bn, var_bn = norm_forward(X, gamma, beta, axis=0)
Y_ln, cache_ln, mean_ln, var_ln = norm_forward(X, gamma, beta, axis=1)

assert X.shape == Y_bn.shape == Y_ln.shape == (B, H)
assert mean_bn.shape == var_bn.shape == (1, H)
assert mean_ln.shape == var_ln.shape == (B, 1)

### Invariantes por eixo

Com `eps > 0`, a média normalizada continua numericamente zero, enquanto a variância fica muito próxima — mas não necessariamente igual — a um.

In [ ]:
bn_mean_err = float(np.max(np.abs(Y_bn.mean(axis=0))))
bn_var_err = float(np.max(np.abs(Y_bn.var(axis=0) - 1.0)))
ln_mean_err = float(np.max(np.abs(Y_ln.mean(axis=1))))
ln_var_err = float(np.max(np.abs(Y_ln.var(axis=1) - 1.0)))

print(f"BatchNorm: erro média={bn_mean_err:.3e}, erro variância={bn_var_err:.3e}")
print(f"LayerNorm: erro média={ln_mean_err:.3e}, erro variância={ln_var_err:.3e}")
assert bn_mean_err < 1e-12
assert ln_mean_err < 1e-12
assert bn_var_err < 1e-3
assert ln_var_err < 1e-3

## 4. Ganho e deslocamento aprendíveis

Depois da normalização, `gamma` e `beta` permitem média e escala diferentes por unidade.

In [ ]:
gamma_affine = np.array([[0.5, 1.0, 1.5, 2.0, 2.5]])
beta_affine = np.array([[-2.0, -1.0, 0.0, 1.0, 2.0]])
Y_affine, _, _, _ = norm_forward(X, gamma_affine, beta_affine, axis=0)

affine_mean_err = float(np.max(np.abs(Y_affine.mean(axis=0, keepdims=True) - beta_affine)))
affine_std_err = float(
    np.max(np.abs(Y_affine.std(axis=0, keepdims=True) - gamma_affine))
)
print("erro da média em relação a beta:", f"{affine_mean_err:.3e}")
print("erro do desvio em relação a gamma:", f"{affine_std_err:.3e}")
assert affine_mean_err < 1e-12
assert affine_std_err < 2e-4

## 5. Dependência do lote e lote unitário

O mesmo exemplo-âncora será colocado em dois lotes diferentes. LayerNorm deve permanecer idêntica; BatchNorm em treino não.

In [ ]:
anchor = np.array([[0.5, -1.0, 2.0, 4.0, -0.5]])
companions_a = rng.normal(0, 1, size=(7, H))
companions_b = rng.normal(8, 3, size=(7, H))
batch_a = np.vstack([anchor, companions_a])
batch_b = np.vstack([anchor, companions_b])

anchor_bn_a = norm_forward(batch_a, gamma, beta, axis=0)[0][0]
anchor_bn_b = norm_forward(batch_b, gamma, beta, axis=0)[0][0]
anchor_ln_a = norm_forward(batch_a, gamma, beta, axis=1)[0][0]
anchor_ln_b = norm_forward(batch_b, gamma, beta, axis=1)[0][0]

batch_dependency = float(np.max(np.abs(anchor_bn_a - anchor_bn_b)))
layer_independence = float(np.max(np.abs(anchor_ln_a - anchor_ln_b)))
singleton_bn = norm_forward(anchor, gamma, beta, axis=0)[0]

print("mudança da âncora em BatchNorm:", f"{batch_dependency:.6f}")
print("mudança da âncora em LayerNorm:", f"{layer_independence:.3e}")
print("BatchNorm com B=1:", singleton_bn)
assert batch_dependency > 0.5
assert layer_independence == 0.0
assert np.allclose(singleton_bn, beta, atol=1e-12)
assert not np.allclose(anchor_ln_a, beta)

## 6. BatchNorm com estado explícito

Adotamos `running ← rho × running + (1-rho) × batch`. Treino atualiza o estado; inferência apenas o lê.

In [ ]:
class BatchNormDense:
    def __init__(self, features, rho=0.9, eps=1e-5):
        self.gamma = np.ones((1, features), dtype=np.float64)
        self.beta = np.zeros((1, features), dtype=np.float64)
        self.running_mean = np.zeros((1, features), dtype=np.float64)
        self.running_var = np.ones((1, features), dtype=np.float64)
        self.rho = float(rho)
        self.eps = float(eps)

    def forward(self, X, training):
        if training:
            mean = X.mean(axis=0, keepdims=True)
            var = X.var(axis=0, keepdims=True)
            self.running_mean = self.rho * self.running_mean + (1 - self.rho) * mean
            self.running_var = self.rho * self.running_var + (1 - self.rho) * var
        else:
            mean, var = self.running_mean, self.running_var
        inv_std = 1.0 / np.sqrt(var + self.eps)
        xhat = (X - mean) * inv_std
        cache = {"xhat": xhat, "inv_std": inv_std, "axis": 0}
        return self.gamma * xhat + self.beta, cache


bn_layer = BatchNormDense(H, rho=0.9)
before_mean = bn_layer.running_mean.copy()
_ = bn_layer.forward(X, training=True)
assert not np.array_equal(bn_layer.running_mean, before_mean)

frozen_mean = bn_layer.running_mean.copy()
frozen_var = bn_layer.running_var.copy()
_ = bn_layer.forward(X[:2], training=False)
assert np.array_equal(bn_layer.running_mean, frozen_mean)
assert np.array_equal(bn_layer.running_var, frozen_var)

### Calibração das estatísticas correntes

Transmitimos 500 lotes de uma distribuição conhecida. Depois, a inferência deve ser determinística e independente dos companheiros.

In [ ]:
rng_stream = np.random.default_rng(SEED + 2)
true_mean = np.array([[-1.0, 0.5, 3.0, 2.0, -2.0]])
true_std = np.array([[2.0, 0.5, 1.5, 1.0, 3.0]])
bn_stream = BatchNormDense(H, rho=0.9)

for _ in range(500):
    stream_batch = rng_stream.normal(true_mean, true_std, size=(128, H))
    bn_stream.forward(stream_batch, training=True)

running_mean_err = float(np.max(np.abs(bn_stream.running_mean - true_mean)))
running_var_err = float(np.max(np.abs(bn_stream.running_var - true_std**2)))
eval_a = bn_stream.forward(batch_a, training=False)[0][0]
eval_b = bn_stream.forward(batch_b, training=False)[0][0]
eval_independence = float(np.max(np.abs(eval_a - eval_b)))

print("erro máximo da média corrente:", f"{running_mean_err:.6f}")
print("erro máximo da variância corrente:", f"{running_var_err:.6f}")
print("mudança da âncora em eval:", f"{eval_independence:.3e}")
assert running_mean_err < 0.2
assert running_var_err < 0.7
assert eval_independence == 0.0

## 7. Backward vetorizado

As somas de `dX` usam o eixo normalizado. Já `dgamma` e `dbeta` sempre somam exemplos (`axis=0`) porque os parâmetros são compartilhados pelo lote.

In [ ]:
def norm_backward(dY, cache, gamma):
    xhat = cache["xhat"]
    inv_std = cache["inv_std"]
    axis = cache["axis"]
    dXhat = dY * gamma
    m = dY.shape[axis]
    sum_d = dXhat.sum(axis=axis, keepdims=True)
    sum_dxhat = (dXhat * xhat).sum(axis=axis, keepdims=True)
    dX = inv_std * (m * dXhat - sum_d - xhat * sum_dxhat) / m
    dgamma = (dY * xhat).sum(axis=0, keepdims=True)
    dbeta = dY.sum(axis=0, keepdims=True)
    return dX, dgamma, dbeta


dY = rng.normal(size=(B, H))
dX_bn, dgamma_bn, dbeta_bn = norm_backward(dY, cache_bn, gamma)
dX_ln, dgamma_ln, dbeta_ln = norm_backward(dY, cache_ln, gamma)

assert dX_bn.shape == dX_ln.shape == (B, H)
assert dgamma_bn.shape == dbeta_bn.shape == (1, H)
assert dgamma_ln.shape == dbeta_ln.shape == (1, H)
assert np.max(np.abs(dX_bn.sum(axis=0))) < 1e-12
assert np.max(np.abs(dX_ln.sum(axis=1))) < 1e-12

### Utilitário de diferenças centrais

Cada coordenada é perturbada em `float64`. O estado não é atualizado durante esses checks.

In [ ]:
def numerical_gradient(array, objective, h=1e-5):
    grad = np.zeros_like(array, dtype=np.float64)
    for idx in np.ndindex(array.shape):
        old = array[idx]
        array[idx] = old + h
        plus = objective()
        array[idx] = old - h
        minus = objective()
        array[idx] = old
        grad[idx] = (plus - minus) / (2 * h)
    return grad


def relative_error(a, b):
    return float(np.max(np.abs(a - b) / np.maximum(1e-12, np.abs(a) + np.abs(b))))


X_gc = rng.normal(size=(6, 4))
gamma_gc = rng.normal(1.0, 0.2, size=(1, 4))
beta_gc = rng.normal(0.0, 0.2, size=(1, 4))
dY_gc = rng.normal(size=(6, 4))
assert np.all(np.isfinite(X_gc))

### Gradient check da BatchNorm

O objetivo escalar é $L=\sum Y\odot dY$.

In [ ]:
def objective_bn():
    return float(np.sum(norm_forward(X_gc, gamma_gc, beta_gc, axis=0)[0] * dY_gc))


_, cache_gc_bn, _, _ = norm_forward(X_gc, gamma_gc, beta_gc, axis=0)
dx_bn_a, dg_bn_a, db_bn_a = norm_backward(dY_gc, cache_gc_bn, gamma_gc)
dx_bn_n = numerical_gradient(X_gc, objective_bn)
dg_bn_n = numerical_gradient(gamma_gc, objective_bn)
db_bn_n = numerical_gradient(beta_gc, objective_bn)

bn_grad_errors = {
    "dX": relative_error(dx_bn_a, dx_bn_n),
    "dgamma": relative_error(dg_bn_a, dg_bn_n),
    "dbeta": relative_error(db_bn_a, db_bn_n),
}
print("erros relativos BatchNorm:", bn_grad_errors)
assert max(bn_grad_errors.values()) < 1e-8

### Gradient check da LayerNorm

Só o eixo das estatísticas e das somas internas de `dX` muda.

In [ ]:
def objective_ln():
    return float(np.sum(norm_forward(X_gc, gamma_gc, beta_gc, axis=1)[0] * dY_gc))


_, cache_gc_ln, _, _ = norm_forward(X_gc, gamma_gc, beta_gc, axis=1)
dx_ln_a, dg_ln_a, db_ln_a = norm_backward(dY_gc, cache_gc_ln, gamma_gc)
dx_ln_n = numerical_gradient(X_gc, objective_ln)
dg_ln_n = numerical_gradient(gamma_gc, objective_ln)
db_ln_n = numerical_gradient(beta_gc, objective_ln)

ln_grad_errors = {
    "dX": relative_error(dx_ln_a, dx_ln_n),
    "dgamma": relative_error(dg_ln_a, dg_ln_n),
    "dbeta": relative_error(db_ln_a, db_ln_n),
}
print("erros relativos LayerNorm:", ln_grad_errors)
assert max(ln_grad_errors.values()) < 1e-8

## 8. O papel de epsilon

Em um grupo quase constante, `eps` domina o denominador. O resultado deve permanecer finito; não se deve exigir variância exatamente um.

In [ ]:
rng_eps = np.random.default_rng(SEED + 3)
X_near_constant = 7.0 + rng_eps.normal(0, 1e-9, size=(32, 4))
Y_eps_small = norm_forward(X_near_constant, np.ones((1, 4)), np.zeros((1, 4)), 0, 1e-16)[0]
Y_eps_safe = norm_forward(X_near_constant, np.ones((1, 4)), np.zeros((1, 4)), 0, 1e-5)[0]
safe_rms = float(np.sqrt(np.mean(Y_eps_safe**2)))
small_rms = float(np.sqrt(np.mean(Y_eps_small**2)))

print(f"RMS com eps=1e-16: {small_rms:.6e}")
print(f"RMS com eps=1e-5:  {safe_rms:.6e}")
assert np.all(np.isfinite(Y_eps_small))
assert np.all(np.isfinite(Y_eps_safe))
assert safe_rms < small_rms
assert Y_eps_safe.var() < 1.0

## 9. Treino e inferência não são intercambiáveis na BatchNorm

Um lote pequeno e deslocado produz estatísticas ruidosas no modo treino. Em `eval`, a camada usa o estado congelado. LayerNorm não possui essa bifurcação.

In [ ]:
tiny_shifted = rng_stream.normal(true_mean + 2.0, true_std, size=(2, H))
state_before = (bn_stream.running_mean.copy(), bn_stream.running_var.copy())
Y_eval_tiny = bn_stream.forward(tiny_shifted, training=False)[0]

# Cópia para não contaminar o estado calibrado durante a demonstração.
bn_copy = BatchNormDense(H, rho=bn_stream.rho, eps=bn_stream.eps)
bn_copy.running_mean = bn_stream.running_mean.copy()
bn_copy.running_var = bn_stream.running_var.copy()
Y_train_tiny = bn_copy.forward(tiny_shifted, training=True)[0]
train_eval_gap = float(np.max(np.abs(Y_train_tiny - Y_eval_tiny)))

Y_ln_once = norm_forward(tiny_shifted, gamma, beta, axis=1)[0]
Y_ln_again = norm_forward(tiny_shifted, gamma, beta, axis=1)[0]

print("maior diferença BatchNorm train × eval:", f"{train_eval_gap:.6f}")
assert train_eval_gap > 0.1
assert np.array_equal(bn_stream.running_mean, state_before[0])
assert np.array_equal(bn_stream.running_var, state_before[1])
assert np.array_equal(Y_ln_once, Y_ln_again)

## 10. Auditoria final

Cada grupo abaixo representa um contrato independente: dados, shapes, eixos, estado, gradientes ou estabilidade.

In [ ]:
audit = {
    "padronizador ajustado no treino": train_mean_err < 1e-12 and train_var_err < 1e-7,
    "contraprova de leakage": leakage_shift > 0.5,
    "condicionamento bruto alto": cond_raw > 1e5,
    "condicionamento padronizado": cond_std < 1.3,
    "ganho no teste reservado": test_mse_raw / test_mse_std > 100,
    "shapes BatchNorm": mean_bn.shape == (1, H),
    "shapes LayerNorm": mean_ln.shape == (B, 1),
    "invariantes BatchNorm": bn_mean_err < 1e-12 and bn_var_err < 1e-3,
    "invariantes LayerNorm": ln_mean_err < 1e-12 and ln_var_err < 1e-3,
    "gamma e beta": affine_mean_err < 1e-12 and affine_std_err < 2e-4,
    "dependência do lote": batch_dependency > 0.5 and layer_independence == 0.0,
    "lote unitário": np.allclose(singleton_bn, beta, atol=1e-12),
    "estatísticas correntes": running_mean_err < 0.2 and running_var_err < 0.7,
    "eval independente do lote": eval_independence == 0.0,
    "backward BatchNorm": max(bn_grad_errors.values()) < 1e-8,
    "backward LayerNorm": max(ln_grad_errors.values()) < 1e-8,
    "epsilon finito": np.all(np.isfinite(Y_eps_safe)) and safe_rms < small_rms,
    "train versus eval": train_eval_gap > 0.1,
}

for name, ok in audit.items():
    print(f"[{'OK' if ok else 'FALHOU'}] {name}")

assert all(audit.values())
print(f"\n{sum(audit.values())}/{len(audit)} grupos de auditoria aprovados.")